# RESPIRA - AI-Assisted Chest X-Ray Analysis
## Methodology Fusion Model Training on Google Colab (T4 / A100 GPU)

This notebook trains the full dual-backbone architecture strictly adhering to the RESPIRA methodology diagram:
- **Dual Backbones**: EfficientNet-B0 (spatial maps) + ViT-B/16 (1024 visual tokens at 512×512)
- **Bidirectional Cross-Attention**: CNN $\leftrightarrow$ ViT feature exchange
- **Disease-Conditioned Attention**: 6 dedicated disease query vectors
- **Disease Relationship Modeling**: Self-Attention Adjacency Matrix + Graph Convolutional Network (GCN)
- **Dedicated Multi-Label Heads** + **Monte Carlo (MC) Dropout** for epistemic uncertainty estimation

### Step 1: Verify GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (or A100/L4) via `Runtime > Change runtime type > T4 GPU`.

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive (Optional for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 3: Install Required Libraries

In [ ]:
!pip install -q timm scikit-learn matplotlib pandas seaborn

### Step 4: Extract Codebase and Preprocessed Dataset
Upload `Lung_Disease_Preprocessed_512.zip` and the repository to your Drive or Colab storage.

In [ ]:
# Example: Unzip dataset directly into ./data/
!mkdir -p data
!unzip -q /content/drive/MyDrive/Lung_Disease_Preprocessed_512.zip -d data/

### Step 5: Launch Methodology Model Training
- Resume from existing checkpoint (Epoch 25 -> 55)
- Uses Multi-Class Label-Smoothed Cross-Entropy Loss to resolve inter-class confusion
- On Colab T4 GPU (16 GB VRAM), use `--batch_size 16`

In [ ]:
!python scripts/train_methodology_fusion.py \
    --data_dir data/Lung_Disease_Preprocessed_512 \
    --output_dir outputs/methodology_fusion \
    --batch_size 16 \
    --epochs 55 \
    --resume \
    --lr_backbone 1e-5 \
    --lr_fusion 8e-5 \
    --mc_samples 15 \
    --workers 2

### Step 6: Inspect Evaluation Metrics and Training Curves

In [ ]:
from IPython.display import Image, display
display(Image('outputs/methodology_fusion/evaluation/training_curves.png'))
display(Image('outputs/methodology_fusion/evaluation/confusion_matrix.png'))
display(Image('outputs/methodology_fusion/evaluation/uncertainty_distribution.png'))

### Step 7: Backup Best Model to Google Drive

In [ ]:
!cp outputs/methodology_fusion/checkpoints/best_model.pth /content/drive/MyDrive/best_model.pth
print("Model checkpoint saved successfully to Google Drive!")